# Gender Disparities in AP Exams: Mann–Kendall Trend Analysis

**Paper:** *Mapping Gender Disparities in Advanced Academics: Participation and Top Achievement Trends Across Advanced Placement (AP) Exams*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dr-Kaya/AP-Exam-Gender-Disparities/blob/main/AP_Gender_Trend_Analysis.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-black?logo=github)](https://github.com/Dr-Kaya/AP-Exam-Gender-Disparities)

---

## About This Notebook

This notebook fully reproduces the trend analyses reported in the paper. It runs **top-to-bottom with no configuration** — dependencies install automatically in Section 1, and both datasets load directly from GitHub in Section 2.

### Two Datasets, Three Research Questions

| File | Contents | Research Question |
|---|---|---|
| `participation.csv` | MFR-P values by exam and year (1997–2020) | RQ1 |
| `top_achievement.csv` | MFR-TA values by exam and year (1997–2020) | RQ2 |
| Both combined | Spearman correlation between MFR-P and MFR-TA | RQ3 |

### Key Constructs

| Index | Full Name | Interpretation |
|---|---|---|
| **MFR-P** | Male-to-Female Ratio in Participation | > 1.0 = male overrepresentation; < 1.0 = female overrepresentation |
| **MFR-TA** | Male-to-Female Ratio in Top Achievement | MFR of students scoring 5 on AP exam |

> **Gender parity = MFR of 1.0.** A decreasing trend means the gap is closing; an increasing trend means it is widening.

## Section 1 — Install & Import Dependencies

`pymannkendall` is the only package not pre-installed in Colab.

In [ ]:
!pip install pymannkendall --quiet
print('Installation complete.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pymannkendall as mk        # Mann-Kendall test + Sen's slope
from scipy import stats           # Spearman rank correlation
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

print(f'pandas {pd.__version__}  |  numpy {np.__version__}  |  pymannkendall {mk.__version__}')

## Section 2 — Load and Prepare Data

**Raw file structure:** Each CSV has AP exam subjects as rows and years as columns (plus an 'Average' column).  
The preparation pipeline:
1. Transposes so rows = years, columns = exams
2. Keeps Year + 45 AP exam columns
3. Drops the last row (a pre-computed average row, not a real year)
4. Converts all values to numeric

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/Dr-Kaya/AP-Exam-Gender-Disparities/main/'
URL_PAR = BASE_URL + 'participation.csv'
URL_TA  = BASE_URL + 'top_achievement.csv'


def load_and_prepare(url_or_path, label):
    """
    Load an AP gender disparity CSV and prepare it for Mann-Kendall analysis.

    The raw CSV stores exam subjects as rows and years as columns. This function
    transposes the data so each row is a year and each column is an AP exam,
    which is the standard format for time-series analysis. The last row of the
    raw file (a pre-computed average) is dropped before analysis.

    Parameters
    ----------
    url_or_path : str   URL or local file path to the CSV.
    label : str         Display label for logging.

    Returns
    -------
    pd.DataFrame  Rows = years (1997-2020), columns = Year + 45 AP exams.
    """
    df = pd.read_csv(url_or_path)

    # Transpose: exams were rows, years were columns -> flip it
    df2 = df.T
    header = df2.iloc[0]          # After transpose, first row = exam names
    df2 = df2[1:]                 # Remove the header row from the data
    df2.columns = header
    df2.reset_index(inplace=True)
    df2.rename(columns={'index': 'Year'}, inplace=True)

    # Keep Year + 45 exam columns (drops 'Average' and any extra columns)
    df2 = df2[df2.columns[:46]]

    # Drop the last row — it is a pre-computed summary average, not a real year
    df2.drop(index=df2.index[-1], inplace=True)

    # Convert everything to numeric (necessary after string-based transpose)
    df3 = df2.apply(pd.to_numeric, errors='coerce')

    print(f'[{label}] {df3.shape[0]} years x {df3.shape[1]-1} exams  '
          f'| Years: {int(df3["Year"].min())}-{int(df3["Year"].max())}')
    return df3


df_par = load_and_prepare(URL_PAR, 'MFR-P  (participation)')
df_ta  = load_and_prepare(URL_TA,  'MFR-TA (top achievement)')

## Section 3 — Explore the Datasets

In [ ]:
print('MFR-P (participation) — first 5 rows:')
df_par.head()

In [ ]:
print('MFR-TA (top achievement) — first 5 rows:')
df_ta.head()

In [ ]:
# List all 45 AP exam subjects
exam_cols = df_par.columns[1:]
print(f'{len(exam_cols)} AP exam subjects:\n')
for i, col in enumerate(exam_cols, 1):
    print(f'  {i:>2}. {col}')

## Section 4 — Analysis Functions

`run_mann_kendall()` runs the MK test and Sen's slope for one exam time series.  
`run_all_exams()` applies it across all 45 exams in a dataset.

In [ ]:
def run_mann_kendall(series):
    """
    Run the Mann-Kendall trend test and compute Sen's slope for one exam.

    The Mann-Kendall test is non-parametric and rank-based: it detects whether
    an MFR time series has a statistically significant monotonic trend. It
    makes no distributional assumptions and handles missing values gracefully.

    Sen's slope is the median slope across all pairs of data points — it is
    more robust than OLS regression when outliers may be present.

    The intercept is back-calculated from the last observed data point and
    Sen's slope, anchoring the trend line to observed data.

    Zeros are treated as missing data (not truly zero MFR values) and are
    excluded when computing minimum and maximum statistics.

    Returns a dict with: Trend, p, slope (Sen's), z, intercept, Tau, s,
    var_s, minimum, maximum, mean, SD, n.
    """
    clean = series.dropna()

    # Need at least 3 observations for a meaningful MK test
    if len(clean) < 3:
        return {
            'Trend': 'insufficient data', 'p': None, 'slope': None,
            'z': None, 'intercept': None, 'Tau': None, 's': None,
            'var_s': None, 'minimum': None, 'maximum': None,
            'mean': round(float(clean.mean()), 4) if len(clean) > 0 else None,
            'SD': None, 'n': int(len(clean)),
        }

    trend, h, p, z, Tau, s, var_s, slope, intercept = mk.original_test(clean)
    nonzero = series.mask(series == 0)
    return {
        'Trend':     trend,
        'p':         round(p, 4),
        'slope':     round(slope, 4),
        'z':         round(z, 4),
        'intercept': round(intercept, 4),
        'Tau':       round(Tau, 4),
        's':         int(s),
        'var_s':     round(var_s, 4),
        'minimum':   round(float(nonzero.min()), 4),
        'maximum':   round(float(nonzero.max()), 4),
        'mean':      round(float(series.mean()), 4),
        'SD':        round(float(series.std(ddof=1)), 4),
        'n':         int(clean.count()),
    }


def run_all_exams(df, label):
    """Run Mann-Kendall across all exam columns; return a results DataFrame."""
    rows = []
    for col in df.columns[1:]:       # Skip 'Year'
        r = run_mann_kendall(df[col])
        r['Exam'] = col
        rows.append(r)
    results = pd.DataFrame(rows)
    col_order = ['Exam','Trend','p','slope','z','intercept',
                 'Tau','s','var_s','minimum','maximum','mean','SD','n']
    results = results[col_order].reset_index(drop=True)
    sig = (results['p'] < 0.05).sum()
    print(f'[{label}] {len(results)} exams | Significant (p<.05): {sig}')
    return results


print('Functions defined.')

## Section 5 — RQ1: Trends in Gender Disparities in Participation (MFR-P)

**Research Question 1:** How do gender disparities in AP exam participation change over years across subjects?

Results reproduce **Table 3** in the manuscript.

In [ ]:
results_par = run_all_exams(df_par, 'MFR-P')
results_par

In [ ]:
# Significant trends summary
sig = results_par[results_par['p'] < 0.05]
print(f'Significant MFR-P trends: {len(sig)} / {len(results_par)} exams\n')

print(f'Increasing ({len(sig[sig["Trend"]=="increasing"])}): MFR-P rising — male advantage growing')
for _, r in sig[sig['Trend']=='increasing'].iterrows():
    print(f'  {r["Exam"]:<35} slope={r["slope"]:+.4f}  p={r["p"]:.4f}')

print(f'\nDecreasing ({len(sig[sig["Trend"]=="decreasing"])}): MFR-P falling — moving toward parity')
for _, r in sig[sig['Trend']=='decreasing'].iterrows():
    print(f'  {r["Exam"]:<35} slope={r["slope"]:+.4f}  p={r["p"]:.4f}')

## Section 6 — RQ2: Trends in Gender Disparities in Top Achievement (MFR-TA)

**Research Question 2:** How do gender disparities in AP exam top achievement change over years across subjects?

Top achievement = scoring a **5** on the AP exam (College Board's highest score).

Results reproduce **Table 4** in the manuscript.

In [ ]:
results_ta = run_all_exams(df_ta, 'MFR-TA')
results_ta

In [ ]:
sig = results_ta[results_ta['p'] < 0.05]
print(f'Significant MFR-TA trends: {len(sig)} / {len(results_ta)} exams\n')

print(f'Increasing ({len(sig[sig["Trend"]=="increasing"])}): male top-achievement advantage growing')
for _, r in sig[sig['Trend']=='increasing'].iterrows():
    print(f'  {r["Exam"]:<35} slope={r["slope"]:+.4f}  p={r["p"]:.4f}')

print(f'\nDecreasing ({len(sig[sig["Trend"]=="decreasing"])}): female top-achievement growing')
for _, r in sig[sig['Trend']=='decreasing'].iterrows():
    print(f'  {r["Exam"]:<35} slope={r["slope"]:+.4f}  p={r["p"]:.4f}')

## Section 7 — RQ3: Spearman Correlation Between MFR-P and MFR-TA

**Research Question 3:** To what extent are gender disparities in participation and top achievement related?

Spearman's ρ is used because it does not assume linearity and is robust to outliers.

Results reproduce **Table 5** in the manuscript.

In [ ]:
rows = []
for col in df_par.columns[1:]:
    # Pair MFR-P and MFR-TA year-by-year; drop rows where either is missing
    combined = pd.DataFrame({
        'MFR_P':  df_par[col].values,
        'MFR_TA': df_ta[col].values if col in df_ta.columns else np.nan
    }).dropna()

    if len(combined) >= 4:
        rho, p_val = stats.spearmanr(combined['MFR_P'], combined['MFR_TA'])
    else:
        rho, p_val = np.nan, np.nan

    rows.append({
        'Exam':        col,
        'rho':         round(rho, 3) if not np.isnan(rho) else None,
        'p-value':     round(p_val, 3) if not np.isnan(p_val) else None,
        'Significant': bool(p_val < 0.05) if not np.isnan(p_val) else None,
        'n':           len(combined),
    })

spearman_df = pd.DataFrame(rows)
sig_count = spearman_df['Significant'].sum()
print(f'Significant Spearman correlations (p<.05): {sig_count}/{len(spearman_df)}')
spearman_df

## Section 8 — Visualization

In [ ]:
def plot_trend(df, results, exam_name, dataset_label, color='steelblue'):
    """
    Plot MFR time series for one AP exam with Sen's slope trend line overlaid.

    The observed MFR values are plotted as connected dots. The dashed red line
    is the Sen's slope trend line, anchored to the last observed data point.
    The dotted gray line marks MFR = 1.0 (perfect gender parity).

    Parameters
    ----------
    df : pd.DataFrame      Prepared MFR dataset.
    results : pd.DataFrame Results table from run_all_exams().
    exam_name : str        Name of the AP exam to plot.
    dataset_label : str    Label for y-axis (e.g. 'MFR-P' or 'MFR-TA').
    color : str            Color for observed data line.
    """
    if exam_name not in df.columns:
        print(f"'{exam_name}' not found. Valid exams:")
        print(list(df.columns[1:]))
        return

    row = results[results['Exam'] == exam_name].iloc[0]
    x = df['Year']
    y = df[exam_name]
    trend_line = row['slope'] * x + row['intercept']
    p_str = f'p = {row["p"]:.4f}' if row['p'] >= 0.001 else 'p < .001'

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(x, y, '-o', color=color, linewidth=2, markersize=6,
            label=f'Observed {dataset_label}', zorder=3)
    ax.plot(x, trend_line, '--', color='crimson', linewidth=2,
            label=f"Sen's slope = {row['slope']:+.4f}/yr", zorder=2)
    ax.axhline(1.0, color='gray', linewidth=1.2, linestyle=':',
               label='Gender parity (MFR = 1.0)', zorder=1)
    ax.set_title(
        f'{exam_name}  ({dataset_label})\n'
        f'Trend: {row["Trend"]}   |   Sen\'s slope = {row["slope"]:+.4f}   |   {p_str}',
        fontsize=13, fontweight='bold')
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel(f'{dataset_label} Index', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(x.astype(int), rotation=45, ha='right')
    ax.legend(fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

print('Plotting function defined.')

In [ ]:
# Plot participation trend — change exam name to explore any subject
plot_trend(df_par, results_par, 'COMP SCI-A', 'MFR-P', color='steelblue')

In [ ]:
# Plot top achievement trend for the same exam
plot_trend(df_ta, results_ta, 'COMP SCI-A', 'MFR-TA', color='darkorange')

In [ ]:
# Multi-panel grid — edit EXAMS_TO_PLOT to show any exams side by side
EXAMS_TO_PLOT = ['COMP SCI-A', 'CALCULUS AB', 'BIOLOGY', 'PSYCHOLOGY',
                 'STUDIO ART - 2D DESIGN', 'STATISTICS']

n_cols = 2
n_rows = (len(EXAMS_TO_PLOT) + 1) // 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
axes = axes.flatten()
years = df_par['Year']

for ax, exam in zip(axes, EXAMS_TO_PLOT):
    if exam not in df_par.columns:
        ax.set_visible(False)
        continue
    r = results_par[results_par['Exam'] == exam]
    if len(r) == 0:
        ax.set_visible(False)
        continue
    r = r.iloc[0]
    trend_line = r['slope'] * years + r['intercept']
    ax.plot(years, df_par[exam], '-o', color='steelblue', linewidth=1.8, markersize=5)
    ax.plot(years, trend_line, '--', color='crimson', linewidth=2,
            label=f"slope={r['slope']:+.4f}")
    ax.axhline(1.0, color='gray', linewidth=1.2, linestyle=':')
    p_str = f'p={r["p"]:.3f}' if r['p'] >= 0.001 else 'p<.001'
    ax.set_title(f'{exam}\n{r["Trend"]}, {p_str}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Year', fontsize=9)
    ax.set_ylabel('MFR-P', fontsize=9)
    ax.set_xticks(years[::4])
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.4)

for ax in axes[len(EXAMS_TO_PLOT):]:
    ax.set_visible(False)

plt.suptitle('AP Exam Gender Disparity in Participation (MFR-P)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Section 9 — Export Results

Files are saved to Colab instance storage. Use the Files panel (left sidebar) to download, or uncomment `files.download()` lines.

In [ ]:
results_par.to_csv('participation_trends_RQ1.csv', index=False)
results_par.to_excel('participation_trends_RQ1.xlsx', index=False)
print('Saved: participation_trends_RQ1.csv / .xlsx  (Table 3 in manuscript)')

results_ta.to_csv('top_achievement_trends_RQ2.csv', index=False)
results_ta.to_excel('top_achievement_trends_RQ2.xlsx', index=False)
print('Saved: top_achievement_trends_RQ2.csv / .xlsx  (Table 4 in manuscript)')

spearman_df.to_csv('spearman_correlations_RQ3.csv', index=False)
spearman_df.to_excel('spearman_correlations_RQ3.xlsx', index=False)
print('Saved: spearman_correlations_RQ3.csv / .xlsx  (Table 5 in manuscript)')

# Uncomment to trigger automatic download in Colab:
# from google.colab import files
# files.download('participation_trends_RQ1.csv')
# files.download('top_achievement_trends_RQ2.csv')
# files.download('spearman_correlations_RQ3.csv')

---

## Transparency and Openness

This analysis follows the **Transparency and Openness Promotion (TOP) Guidelines**. All data, code, and materials are publicly available at:

> **https://github.com/Dr-Kaya/AP-Exam-Gender-Disparities**

Key packages: `pymannkendall` (MK test + Sen's slope), `scipy.stats` (Spearman correlation), `pandas`, `numpy`, `matplotlib`, `seaborn`.

This study's design and analysis were **not pre-registered**.

---
*Questions? Open a GitHub Issue at the repository above.*